In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai import OpenAI
from typing import Dict, List
import os
from pathlib import Path

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
# Instructions for the Agents

In [4]:
security_instructions = """
You are a SQL security expert. Review SQL code for:
1. SQL injection vulnerabilities (parameterized queries, input validation)
2. Privilege escalation risks (excessive permissions)
3. Sensitive data exposure (proper encryption, masking)
4. Authentication and authorization issues
5. Dangerous operations (DROP, TRUNCATE without safeguards)

Provide specific examples and fixes.|
"""

In [5]:
performance_instructions = """
You are a SQL performance optimization expert. Review SQL code for:
1. Missing indexes on frequently queried columns
2. N+1 query problems
3. Inefficient JOINs (cartesian products, unnecessary joins)
4. SELECT * instead of specific columns
5. Suboptimal WHERE clauses
6. Missing LIMIT clauses on large result sets
7. Unnecessary DISTINCT or GROUP BY

Provide optimized alternatives.
"""

In [6]:
best_practices_instructions = """
You are a SQL best practices expert. Review SQL code for:
1. Code readability and formatting
2. Proper naming conventions
3. Transaction management
4. Error handling
5. Comments and documentation
6. Use of deprecated features
7. Database portability issues
8. Schema design problems (normalization, data types)

Provide improvements.
"""

In [7]:
# Agents using above instructions

In [8]:
security_reviewer = Agent(
    name = "Security Reviewer",
    instructions = security_instructions,
    model = 'gpt-4o'
)

In [9]:
performance_reviewer = Agent(
    name = 'Performance Reviewer',
    instructions = performance_instructions,
    model = 'gpt-4o'
)

In [10]:
best_practices_reviewer = Agent(
    name = 'Best Practices Reviewer',
    instructions = best_practices_instructions,
    model = 'gpt-4o'
)

In [11]:
# function tools

In [12]:
@function_tool
def save_review_report(filename: str, review_content: str):
    output_dir = Path("./review_reports")
    output_dir.mkdir(exist_ok=True)
    filepath = output_dir/filename
    filepath.write_text(review_content)
    return {
        "status": "success",
        "message": f"Review Report saved to path {filepath}"
    }


In [13]:
@function_tool
def analyze_sql_complexity(sql_code: str):
    metrics = {
        "line_count": len(sql_code.split('\n')),
        "join_count": sql_code.upper().count('JOIN'),
        "subquery_count": sql_code.upper().count('SELECT') - 1, # ignore the main SELECT
        "filter_count": sql_code.upper().count('WHERE')
    }
    return metrics

In [14]:
# Handoff Agent for consolidation of reviews and generation of report

In [15]:
consolidation_instructions = """
You are a senior SQL reviewer who reviews the feedback from the security reviewer, performance reviewer and best practices reviewer.
You then consolidate the reviews and create a comprehensive report.
Your exact job in specific detail:

1. Go through the reviews from security, performance, and best practices reviewers
2. Remove duplicate findings
3. Prioritize issues (Critical, High, Medium, Low)
4. Create a clear, actionable final report with:
   - Executive Summary
   - Critical Issues (must fix before merge)
   - High Priority Issues (should fix soon)
   - Recommendations (nice to have)
   - Overall verdict: APPROVE, APPROVE WITH CHANGES, or REJECT
Use markdown formatting for the report.
"""

In [16]:
review_consolidator = Agent(
    name = 'Review Consolidator',
    instructions = consolidation_instructions,
    model = 'gpt-4o',
    tools = [save_review_report],
    handoff_description = "Consolidate the reviews from Security, Performance and Best Practices reviewers, and save it."
)

In [17]:
# The Main SQL reviewer agent

In [18]:
sql_reviewer_instructions = """
    You are the main SQL reviewer.
    Follow these steps:
    1. Use all the 3 specialist reviewers - Security Reviewer, Performance Reviewer and Best Practices Reviewer - in parallel, to get comprehensive reviews.
            a. Security Reviewer will check for SQL injection vulnerabilities, privilege escalation risks, sensitive data exposure, authentication and authorization issues, and dangerous operations.
            b. Performance Reviewer will check for missing indexes, N+1 query problems, inefficient JOINs, SELECT * instead of specific columns, suboptimal WHERE clauses, and missing LIMIT clauses on large result sets.
            c. Best Practices Reviewer will check for code readability, proper naming conventions, transaction management, error handling, comments and documentation, use of deprecated features, database portability issues, and schema design problems.
   2. Handoff the reviews from all 3 specialist reviewers to the Review Consolidator to create a comprehensive report.
   3. Get the SQL complexity metrics, using the analyze_sql_complexity tool.
   4. Create a final report with the consolidated reviews and the SQL complexity metrics, using the save_review_report tool.
""" 

In [19]:
tools_for_sql_reviewer = [
    security_reviewer.as_tool(
        tool_name="security_reviewer",
        tool_description="Review SQL code for security vulnerabilities"
    ),
    performance_reviewer.as_tool(
        tool_name="performance_reviewer",
        tool_description="Review SQL code for performance optimization"
    ),
    best_practices_reviewer.as_tool(
        tool_name="best_practices_reviewer",
        tool_description="Review SQL code for best practices"
    ),
    analyze_sql_complexity
]

In [20]:

sql_review_coordinator = Agent(
    name="SQL Review Coordinator",
    instructions=sql_reviewer_instructions,
    tools=tools_for_sql_reviewer,
    handoffs=[review_consolidator],
    model="gpt-4o"
)

In [24]:
# Testing begins

In [26]:
# Pass the SQL code directly to the SQL reviewer agent

In [21]:
async def review_sql_direct():

    sql_code = """
    SELECT * FROM users u
    JOIN orders o ON u.id = o.user_id
    WHERE u.email = 'test@example.com'
    AND o.total > 100
    """
    
    message = f"Review this SQL query:\n\n```{sql_code}\n```"
    
    with trace("Direct SQL Review"):
        result = await Runner.run(sql_review_coordinator, message)
        print(result.final_output)

In [23]:
await review_sql_direct()

The SQL review report has been saved successfully. If you need any more assistance or modifications, feel free to ask!


In [27]:
# Pass the SQL code to the SQL reviewer agent through a file

In [38]:
async def review_sql_from_file(filepath: str):
    sql_code = Path(filepath).read_text()
    
    message = f"""
    Review the SQL code from file: {filepath}
    ```sql
    {sql_code}
    ```
    """
    
    with trace("File SQL Review"):
        result = await Runner.run(sql_review_coordinator, message)
        print(result.final_output)

In [39]:
await review_sql_from_file("./queries/customer_orders.sql")

The review report has been successfully saved as `customer_orders_review_report.md`. Here is an overview of the final report:

---

# Customer Orders SQL Review Report

## Executive Summary
The SQL query provided for retrieving customer orders with product and category details has been reviewed for security, performance, and best practices. While functionally sound, improvements are necessary for security hardening, query optimization, and adherence to best practices. We recommend implementing these changes before deployment.

## Critical Issues
- **SQL Injection Vulnerability**: Ensure parameterized queries are used if this query is exposed to user inputs. This prevents potential SQL injection attacks.
- **Privileges**: Verify that the executing account has only 'SELECT' permissions to enforce the principle of least privilege.

## High Priority Issues
- **Indexing**: Add indexes on all joined columns and `c.created_at` to improve query performance.
- **Column Selection**: Modify the q